# Maia Lite 355M v3 — Clean Pretraining

Clean, reproducible pretraining pipeline for the Maia Lite base model.

**Fixed architecture**
- GPT-2 Medium–matched decoder-only Transformer
- 354,823,168 parameters
- 24 layers, 16 heads, 1,024 embedding dimension
- 50,257-token vocabulary
- 1,024-token context

**Training budget**
- Chinchilla reference: ~20 tokens/parameter
- Exact reference: 7,096,463,360 tokens
- Batch-aligned target: 7,096,467,456 tokens
- Effective batch: 32 sequences = 32,768 tokens/optimizer step
- Target: 216,567 optimizer steps

This notebook intentionally does **not** reuse the old Maia tokenizer, corpus, model weights, samples, or materialization state. FineMath has been replaced by OpenWebMath.


In [1]:
# 0. INSTALL / MOUNT
# Reproducible environment for the validated notebook API.
# Do not use an unconstrained `pip install -U transformers`.

!pip -q install "transformers==5.17.0" datasets tokenizers accelerate

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# 1. GLOBAL SPECIFICATION

import os
import gc
import json
import math
import time
import random
import hashlib
from pathlib import Path

import numpy as np
import torch

from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    TrainingArguments,
    Trainer,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path("/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3")
TOKENIZER_DIR = PROJECT_DIR / "tokenizer"
TOKENIZER_SAMPLE_DIR = PROJECT_DIR / "tokenizer_sample"
CORPUS_DIR = PROJECT_DIR / "corpus"
TRAIN_DIR = CORPUS_DIR / "train"
STATE_DIR = CORPUS_DIR / "state"
MANIFEST_DIR = CORPUS_DIR / "manifests"
VALIDATION_DIR = PROJECT_DIR / "validation"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

for p in [
    PROJECT_DIR, TOKENIZER_DIR, TOKENIZER_SAMPLE_DIR,
    CORPUS_DIR, TRAIN_DIR, STATE_DIR, MANIFEST_DIR,
    VALIDATION_DIR, CHECKPOINT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

VOCAB_SIZE = 50_257
SEQUENCE_LENGTH = 1_024
N_PARAMS_REFERENCE = 354_823_168
CHINCHILLA_TOKENS = 20 * N_PARAMS_REFERENCE

MICRO_BATCH = 16
GRAD_ACCUM = 2
EFFECTIVE_BATCH = MICRO_BATCH * GRAD_ACCUM
TOKENS_PER_STEP = EFFECTIVE_BATCH * SEQUENCE_LENGTH

TARGET_STEPS = math.ceil(CHINCHILLA_TOKENS / TOKENS_PER_STEP)
TARGET_TOTAL_TOKENS = TARGET_STEPS * TOKENS_PER_STEP

# 8 Mi tokens = 16 MiB in uint16, divisible by 1024.
SHARD_TOKENS = 8_388_608
DTYPE = np.uint16

print(f"Reference parameters:     {N_PARAMS_REFERENCE:,}")
print(f"Chinchilla target:        {CHINCHILLA_TOKENS:,} tokens")
print(f"Tokens/optimizer step:    {TOKENS_PER_STEP:,}")
print(f"Target optimizer steps:   {TARGET_STEPS:,}")
print(f"Batch-aligned target:     {TARGET_TOTAL_TOKENS:,} tokens")
print(f"Difference from 20N:      {TARGET_TOTAL_TOKENS - CHINCHILLA_TOKENS:,} tokens")

Reference parameters:     354,823,168
Chinchilla target:        7,096,463,360 tokens
Tokens/optimizer step:    32,768
Target optimizer steps:   216,567
Batch-aligned target:     7,096,467,456 tokens
Difference from 20N:      4,096 tokens


In [3]:
# 2. CORPUS SPECIFICATION
# Same high-quality source families validated in the previous experiment,
# but rebuilt from raw documents with the NEW tokenizer.
#
# Weights sum to 1.0:
#   EN 45%, PT 22.5%, ES 22.5%, Code 5%, Math 3%, Science 2%.

SOURCES = [
    {
        "id": "fineweb_edu_en",
        "name": "FineWeb-Edu EN",
        "repo": "HuggingFaceFW/fineweb-edu",
        "config": "sample-10BT",
        "split": "train",
        "weight": 0.45,
        "text_field": "text",
    },
    {
        "id": "fineweb2_hq_pt",
        "name": "FineWeb2-HQ PT",
        "repo": "epfml/FineWeb2-HQ",
        "config": "por_Latn",
        "split": "train",
        "weight": 0.225,
        "text_field": "text",
    },
    {
        "id": "fineweb2_hq_es",
        "name": "FineWeb2-HQ ES",
        "repo": "epfml/FineWeb2-HQ",
        "config": "spa_Latn",
        "split": "train",
        "weight": 0.225,
        "text_field": "text",
    },
    {
        "id": "opencoder",
        "name": "OpenCoder FineWeb Code",
        "repo": "OpenCoder-LLM/opc-fineweb-code-corpus",
        "config": None,
        "split": "train",
        "weight": 0.05,
        "text_field": "text",
    },
    {
        "id": "openwebmath",
        "name": "OpenWebMath",
        "repo": "open-web-math/open-web-math",
        "config": None,
        "split": "train",
        "weight": 0.03,
        "text_field": "text",
    },
    {
        "id": "pes2o",
        "name": "peS2o Scientific",
        "repo": "common-pile/peS2o",
        "config": None,
        "split": "train",
        "weight": 0.02,
        "text_field": "text",
    },
]

assert abs(sum(s["weight"] for s in SOURCES) - 1.0) < 1e-12

# Allocate source targets in whole 1024-token sequences.
remaining_sequences = TARGET_TOTAL_TOKENS // SEQUENCE_LENGTH
for i, source in enumerate(SOURCES):
    if i < len(SOURCES) - 1:
        seqs = int((TARGET_TOTAL_TOKENS // SEQUENCE_LENGTH) * source["weight"])
    else:
        seqs = remaining_sequences
    source["target_sequences"] = seqs
    source["target_tokens"] = seqs * SEQUENCE_LENGTH
    remaining_sequences -= seqs

# Correct any rounding remainder by assigning it to English.
if remaining_sequences:
    SOURCES[0]["target_sequences"] += remaining_sequences
    SOURCES[0]["target_tokens"] += remaining_sequences * SEQUENCE_LENGTH

print(f"{'Source':28s} {'Weight':>8s} {'Tokens':>18s}")
for s in SOURCES:
    print(f"{s['name']:28s} {s['weight']:8.3%} {s['target_tokens']:18,}")
print("-" * 58)
print(f"{'TOTAL':28s} {'100%':>8s} {sum(s['target_tokens'] for s in SOURCES):18,}")

Source                         Weight             Tokens
FineWeb-Edu EN                45.000%      3,193,409,536
FineWeb2-HQ PT                22.500%      1,596,704,768
FineWeb2-HQ ES                22.500%      1,596,704,768
OpenCoder FineWeb Code         5.000%        354,823,168
OpenWebMath                    3.000%        212,893,696
peS2o Scientific               2.000%        141,931,520
----------------------------------------------------------
TOTAL                            100%      7,096,467,456


## Tokenizer

Train a fresh byte-level BPE tokenizer from a **stratified sample of the same source families** used for pretraining.

The special tokens are fixed at IDs 0–4. During corpus materialization, `</s>` is appended **explicitly** after every document. We do not rely on `add_special_tokens=True`.


In [4]:
# 3. TOKENIZER TRAINING SAMPLE — LOAD OR BUILD ONCE
#
# IMPORTANT:
# Existing tokenizer sample files are IMMUTABLE.
# A normal notebook restart NEVER rebuilds or overwrites them.
#
# Set FORCE_REBUILD_TOKENIZER_SAMPLES=True only if you intentionally want
# to create a completely new tokenizer experiment.

TOKENIZER_SAMPLE_BYTES = 1_073_741_824  # 1 GiB
MIN_DOCUMENT_CHARS = 200
FORCE_REBUILD_TOKENIZER_SAMPLES = False

sample_files = []

for source in SOURCES:
    target_bytes = int(TOKENIZER_SAMPLE_BYTES * source["weight"])
    out_path = TOKENIZER_SAMPLE_DIR / f"{source['id']}.txt"
    sample_files.append(str(out_path))

    # Existing sample is authoritative. Do NOT compare against target_bytes:
    # the original builder intentionally stopped at a document boundary, so
    # valid files can be slightly smaller than the nominal byte target.
    if out_path.exists() and out_path.stat().st_size > 1_000_000:
        if not FORCE_REBUILD_TOKENIZER_SAMPLES:
            print(
                f"[FROZEN] {source['name']}: "
                f"{out_path.stat().st_size / 1024**2:.2f} MiB"
            )
            continue

    if out_path.exists() and not FORCE_REBUILD_TOKENIZER_SAMPLES:
        raise RuntimeError(
            f"Tokenizer sample exists but is suspiciously small: {out_path}\n"
            "Refusing to overwrite it automatically."
        )

    if FORCE_REBUILD_TOKENIZER_SAMPLES and out_path.exists():
        print(f"[FORCE] Removing tokenizer sample: {out_path}")
        out_path.unlink()

    print(f"\nBuilding tokenizer sample: {source['name']}")
    print(f"Target: {target_bytes:,} bytes")

    ds = load_dataset(
        source["repo"],
        source["config"],
        split=source["split"],
        streaming=True,
    )

    written = 0
    documents = 0
    temp_path = Path(str(out_path) + ".tmp")

    if temp_path.exists():
        temp_path.unlink()

    with open(temp_path, "w", encoding="utf-8") as f:
        for row in ds:
            text = row.get(source["text_field"])
            if not isinstance(text, str):
                continue

            text = text.strip()
            if len(text) < MIN_DOCUMENT_CHARS:
                continue

            encoded = (text + "\n").encode("utf-8")

            # Stop at a document boundary. This is why the final file can be
            # slightly smaller than target_bytes.
            if written + len(encoded) > target_bytes:
                break

            f.write(text)
            f.write("\n")
            written += len(encoded)
            documents += 1

            if documents % 10_000 == 0:
                print(
                    f"  documents={documents:,} bytes={written:,}",
                    end="\r",
                )

    if temp_path.stat().st_size <= 1_000_000:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError(
            f"Tokenizer sample build failed or produced a tiny file: {out_path}"
        )

    os.replace(temp_path, out_path)

    print(
        f"  DONE documents={documents:,} "
        f"bytes={out_path.stat().st_size:,}"
    )

print("\nTokenizer sample policy: FROZEN / NON-DESTRUCTIVE")

[FROZEN] FineWeb-Edu EN: 460.68 MiB
[FROZEN] FineWeb2-HQ PT: 230.40 MiB
[FROZEN] FineWeb2-HQ ES: 230.40 MiB
[FROZEN] OpenCoder FineWeb Code: 51.20 MiB
[FROZEN] OpenWebMath: 30.71 MiB
[FROZEN] peS2o Scientific: 20.46 MiB

Tokenizer sample policy: FROZEN / NON-DESTRUCTIVE


In [5]:
# 3B. TOKENIZER SAMPLE PREFLIGHT
print("=" * 80)
print("TOKENIZER SAMPLE PREFLIGHT")
print("=" * 80)
total_bytes = 0
for filename in sample_files:
    path = Path(filename)
    assert path.exists(), f"Missing file: {path}"
    size = path.stat().st_size
    assert size > 1_000_000, f"Suspiciously small sample: {path}"
    total_bytes += size
    with open(path, "r", encoding="utf-8") as f:
        preview = f.read(600)
    print(f"\n{path.name}: {size / 1024**2:.2f} MiB")
    print(repr(preview))
print(f"\nTotal sample: {total_bytes / 1024**2:.2f} MiB")
assert total_bytes > 900 * 1024**2
print("PREFLIGHT PASSED")

TOKENIZER SAMPLE PREFLIGHT

fineweb_edu_en.txt: 460.68 MiB
'The Independent Jane\nFor all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.\nElizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.\nThe freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and knowing her father would disapprove was unusual even for Austen. In her last book Anne Ell'

fineweb2_hq_pt.txt: 230.40 MiB
'Boehner abre as portas para aumentos tributários nos EUA\nPor David Lawder e Mark Felsenthal\nWASHINGTON, 17 Dez (Reuters) - O primeiro movimento real nas negociações sobre o "abismo fiscal" nos Estados Unidos começou no domingo, quando o presidente republicano da Câmara dos Deputados, John Boehner, s

In [6]:
# ==============================================================================
# 4. LOAD FROZEN TOKENIZER OR RECOVER IT SAFELY
# ==============================================================================
#
# NORMAL RESTART:
#   - Load the existing tokenizer from Google Drive.
#   - Audit exact vocabulary size, special-token IDs, and known token IDs.
#   - NEVER delete or retrain anything.
#
# RECOVERY:
#   - If the tokenizer is missing/corrupt, train a candidate in /content.
#   - Verify the candidate BEFORE publishing it to Google Drive.
#   - The existing corpus is NEVER modified by this cell.
#
# The known-ID regression tests below were recorded from the tokenizer that
# produced the materialized Maia Lite v3 corpus.
# ==============================================================================

import json
import shutil
from pathlib import Path

from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2TokenizerFast

SPECIAL_TOKENS = [
    "<s>",
    "<pad>",
    "</s>",
    "<unk>",
    "<mask>",
]

KNOWN_TOKENIZATION = {
    "The theory of general relativity describes gravity.": [
        621, 4388, 303, 1432, 42874, 9500, 14767, 18
    ],
    "A inteligência artificial pode auxiliar a pesquisa científica.": [
        37, 15964, 7965, 1282, 19442, 261, 7151, 11490, 18
    ],
    "La inteligencia artificial es un campo de la informática.": [
        1403, 15489, 7965, 412, 360, 4316, 281, 344, 33713, 18
    ],
    "def square(x): return x * x": [
        9296, 7501, 12, 92, 2755, 3069, 2308, 4333, 2308
    ],
    "E = mc²": [
        41, 1381, 285, 71, 17471
    ],
}

def audit_tokenizer(candidate, label="tokenizer"):
    assert len(candidate) == VOCAB_SIZE, (
        f"{label}: vocabulary mismatch: {len(candidate):,} != {VOCAB_SIZE:,}"
    )
    assert candidate.bos_token_id == 0
    assert candidate.pad_token_id == 1
    assert candidate.eos_token_id == 2
    assert candidate.unk_token_id == 3
    assert candidate.mask_token_id == 4

    for text, expected_ids in KNOWN_TOKENIZATION.items():
        actual_ids = candidate.encode(
            text,
            add_special_tokens=False,
        )
        assert actual_ids == expected_ids, (
            f"{label}: tokenization regression for:\n{text!r}\n"
            f"Expected: {expected_ids}\n"
            f"Actual:   {actual_ids}"
        )

        decoded = candidate.decode(
            actual_ids,
            clean_up_tokenization_spaces=False,
        )
        assert decoded == text, (
            f"{label}: round-trip regression for {text!r}"
        )

    eos_test = candidate.encode(
        "This is document one.",
        add_special_tokens=False,
    )
    eos_test.append(candidate.eos_token_id)
    assert eos_test[-1] == 2

    return True


def load_frozen_tokenizer():
    required = [
        TOKENIZER_DIR / "tokenizer.json",
        TOKENIZER_DIR / "tokenizer_config.json",
        TOKENIZER_DIR / "special_tokens_map.json",
    ]

    if not all(p.exists() for p in required):
        return None

    try:
        candidate = GPT2TokenizerFast.from_pretrained(
            str(TOKENIZER_DIR),
            local_files_only=True,
        )
        audit_tokenizer(candidate, "frozen tokenizer")
        return candidate
    except Exception as exc:
        print("[WARNING] Existing tokenizer failed audit:")
        print(repr(exc))
        return None


tokenizer = load_frozen_tokenizer()

if tokenizer is not None:
    print("=" * 80)
    print("FROZEN TOKENIZER LOADED")
    print("=" * 80)
    print(f"Path:       {TOKENIZER_DIR}")
    print(f"Vocabulary: {len(tokenizer):,}")
    print(f"BOS/PAD/EOS/UNK/MASK: "
          f"{tokenizer.bos_token_id}/"
          f"{tokenizer.pad_token_id}/"
          f"{tokenizer.eos_token_id}/"
          f"{tokenizer.unk_token_id}/"
          f"{tokenizer.mask_token_id}")
    print("Known-ID regression: PASS")
    print("No tokenizer files were modified.")

else:
    print("=" * 80)
    print("TOKENIZER RECOVERY REQUIRED")
    print("=" * 80)
    print("The frozen tokenizer is missing or invalid.")
    print("Training a candidate in LOCAL TEMPORARY STORAGE.")
    print("The corpus and Drive tokenizer directory will not be touched")
    print("until the candidate passes the exact known-ID regression tests.")

    for filename in sample_files:
        path = Path(filename)
        assert path.exists(), f"Missing tokenizer sample: {path}"
        assert path.stat().st_size > 1_000_000, (
            f"Suspicious tokenizer sample: {path}"
        )

    candidate_dir = Path("/content/maia_lite_v3_tokenizer_candidate")

    if candidate_dir.exists():
        shutil.rmtree(candidate_dir)

    candidate_dir.mkdir(parents=True, exist_ok=True)

    tokenizer_raw = ByteLevelBPETokenizer()

    tokenizer_raw.train(
        files=[str(x) for x in sample_files],
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        show_progress=True,
    )

    raw_vocab = tokenizer_raw.get_vocab()

    assert len(raw_vocab) == VOCAB_SIZE
    assert tokenizer_raw.token_to_id("<s>") == 0
    assert tokenizer_raw.token_to_id("<pad>") == 1
    assert tokenizer_raw.token_to_id("</s>") == 2
    assert tokenizer_raw.token_to_id("<unk>") == 3
    assert tokenizer_raw.token_to_id("<mask>") == 4

    tokenizer_raw.save_model(str(candidate_dir))
    tokenizer_raw._tokenizer.save(
        str(candidate_dir / "tokenizer.json")
    )

    candidate = GPT2TokenizerFast(
        tokenizer_file=str(candidate_dir / "tokenizer.json"),
        bos_token="<s>",
        pad_token="<pad>",
        eos_token="</s>",
        unk_token="<unk>",
        mask_token="<mask>",
        model_max_length=SEQUENCE_LENGTH,
    )

    candidate.save_pretrained(str(candidate_dir))

    candidate = GPT2TokenizerFast.from_pretrained(
        str(candidate_dir),
        local_files_only=True,
    )

    # CRITICAL GATE:
    # Do not publish a regenerated tokenizer unless it is token-ID compatible
    # with the tokenizer that produced the existing binary corpus.
    audit_tokenizer(candidate, "recovered candidate")

    print()
    print("Candidate tokenizer EXACT-ID regression: PASS")
    print("Publishing verified tokenizer to Google Drive...")

    TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

    for path in candidate_dir.iterdir():
        if path.is_file():
            temp_dest = TOKENIZER_DIR / (path.name + ".new")
            shutil.copyfile(path, temp_dest)
            os.replace(temp_dest, TOKENIZER_DIR / path.name)

    tokenizer = GPT2TokenizerFast.from_pretrained(
        str(TOKENIZER_DIR),
        local_files_only=True,
    )

    audit_tokenizer(tokenizer, "published tokenizer")

    print("=" * 80)
    print("TOKENIZER RECOVERY COMPLETE")
    print("=" * 80)
    print(f"Vocabulary: {len(tokenizer):,}")
    print("Known-ID regression: PASS")
    print("Existing binary corpus remains untouched.")

TOKENIZER RECOVERY REQUIRED
The frozen tokenizer is missing or invalid.
Training a candidate in LOCAL TEMPORARY STORAGE.
The corpus and Drive tokenizer directory will not be touched
until the candidate passes the exact known-ID regression tests.

Candidate tokenizer EXACT-ID regression: PASS
Publishing verified tokenizer to Google Drive...
TOKENIZER RECOVERY COMPLETE
Vocabulary: 50,257
Known-ID regression: PASS
Existing binary corpus remains untouched.


In [7]:
# 5. TOKENIZER AUDIT
# Critical regression test: explicit EOS must exist in the sequence.

TEST_TEXTS = [
    "Water is a chemical compound composed of hydrogen and oxygen.",
    "The theory of general relativity describes gravity as spacetime curvature.",
    "A computer operating system manages hardware and software resources.",
    "O Brasil é o maior país da América do Sul.",
    "La fotosíntesis convierte la energía luminosa en energía química.",
    "def square(x):\n    return x * x",
]

for text in TEST_TEXTS:
    ids = tokenizer.encode(text, add_special_tokens=False)
    ids_with_eos = ids + [tokenizer.eos_token_id]
    decoded = tokenizer.decode(ids, clean_up_tokenization_spaces=False)

    print("\nTEXT:", repr(text))
    print("TOKENS:", len(ids))
    print("ROUND TRIP:", decoded == text)
    print("LAST TOKEN WITH EXPLICIT EOS:", ids_with_eos[-1], tokenizer.convert_ids_to_tokens([ids_with_eos[-1]]))

    assert decoded == text
    assert ids_with_eos[-1] == tokenizer.eos_token_id

print("\nTokenizer audit: PASS")


TEXT: 'Water is a chemical compound composed of hydrogen and oxygen.'
TOKENS: 11
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'The theory of general relativity describes gravity as spacetime curvature.'
TOKENS: 12
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'A computer operating system manages hardware and software resources.'
TOKENS: 10
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'O Brasil é o maior país da América do Sul.'
TOKENS: 11
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'La fotosíntesis convierte la energía luminosa en energía química.'
TOKENS: 13
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'def square(x):\n    return x * x'
TOKENS: 11
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

Tokenizer audit: PASS


## Corpus materialization

Each source is materialized independently as deterministic `uint16` shards.

Rules:
1. Read complete documents from the source stream.
2. Tokenize without automatic special tokens.
3. Append EOS explicitly to every accepted document.
4. Concatenate document token streams continuously.
5. Write aligned 1024-token sequences.
6. Never pad training data.
7. Persist document counters and SHA-256 manifests for safe resume.

Streaming is used only as an **input transport** while constructing the fixed corpus. The Trainer never trains directly from a streaming dataset.


In [8]:
# 6. CORPUS MATERIALIZER — CRASH-SAFE CARRY
# Training and validation are separated by a deterministic SHA-256 partition.

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def is_validation_document(text):
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big") % 1000 < 5  # ~0.5%

def snapshot_path(source):
    return STATE_DIR / f"{source['id']}_snapshot.npz"

def manifest_path(source):
    return MANIFEST_DIR / f"{source['id']}.json"

def default_state():
    return {
        "documents_seen": 0,
        "documents_accepted": 0,
        "validation_documents_skipped": 0,
        "tokens_written": 0,
        "shards_written": 0,
        "records": [],
        "complete": False,
    }

def load_snapshot(source):
    p = snapshot_path(source)
    if not p.exists():
        return default_state(), np.empty(0, dtype=DTYPE)
    with np.load(p, allow_pickle=False) as z:
        state_json = bytes(z["state_json"].tolist()).decode("utf-8")
        state = json.loads(state_json)
        carry = np.asarray(z["carry"], dtype=DTYPE)
    return state, carry

def save_snapshot(source, state, carry):
    # State + carry are committed as ONE atomic file. A crash can therefore
    # never pair an old state with a new carry (or vice versa).
    p = snapshot_path(source)
    tmp = Path(str(p) + ".tmp.npz")
    state_bytes = json.dumps(state, separators=(",", ":")).encode("utf-8")
    np.savez(
        tmp,
        state_json=np.frombuffer(state_bytes, dtype=np.uint8),
        carry=np.asarray(carry, dtype=DTYPE),
    )
    os.replace(tmp, p)

def atomic_json(path, obj):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    os.replace(tmp, path)

def materialize_source(source):
    state, carry = load_snapshot(source)
    # Manifest is derived from the authoritative snapshot and can be regenerated.
    atomic_json(manifest_path(source), state.get("records", []))
    if state["complete"]:
        print(f"[SKIP] {source['name']} already complete."); return
    print("\n"+"="*80); print(source["name"]); print("="*80)
    print(f"Target tokens: {source['target_tokens']:,}")
    print(f"Resume documents_seen: {state['documents_seen']:,}")
    print(f"Resume tokens_written: {state['tokens_written']:,}")

    ds=load_dataset(source["repo"],source["config"],split=source["split"],streaming=True)
    iterator=iter(ds)
    for _ in range(state["documents_seen"]):
        try: next(iterator)
        except StopIteration: raise RuntimeError(f"Source exhausted while resuming {source['name']}")

    # Carry came from the same atomic snapshot as the state.
    buffer=[carry] if len(carry) else []; buffer_tokens=len(carry)
    while state["tokens_written"] < source["target_tokens"]:
        remaining=source["target_tokens"]-state["tokens_written"]
        needed=min(SHARD_TOKENS,remaining)
        assert needed % SEQUENCE_LENGTH == 0
        while buffer_tokens < needed:
            try: row=next(iterator)
            except StopIteration: raise RuntimeError(f"Source exhausted before target: {source['name']}")
            state["documents_seen"] += 1
            text=row.get(source["text_field"])
            if not isinstance(text,str): continue
            text=text.strip()
            if not text: continue
            if is_validation_document(text):
                state["validation_documents_skipped"] += 1; continue
            ids=tokenizer.encode(text,add_special_tokens=False)
            if not ids: continue
            ids.append(tokenizer.eos_token_id)
            arr=np.asarray(ids,dtype=DTYPE); buffer.append(arr); buffer_tokens += len(arr)
            state["documents_accepted"] += 1

        joined=np.concatenate(buffer); shard=joined[:needed]; surplus=joined[needed:]
        shard_index=state["shards_written"]
        path=TRAIN_DIR/f"{source['id']}_{shard_index:05d}.bin"; tmp=Path(str(path)+".tmp")
        shard.tofile(tmp); os.replace(tmp,path)
        record={"shard":path.name,"tokens":int(needed),"sha256":sha256_file(path)}
        state["records"] = state.get("records",[]) + [record]
        state["tokens_written"] += int(needed); state["shards_written"] += 1
        # Commit authoritative resume snapshot first. If the process dies before
        # the human-readable manifest update, resume regenerates the manifest.
        save_snapshot(source, state, surplus)
        atomic_json(manifest_path(source), state["records"])
        buffer=[surplus] if len(surplus) else []; buffer_tokens=len(surplus)
        print(f"shard={shard_index:05d} tokens={state['tokens_written']:,}/{source['target_tokens']:,} docs={state['documents_seen']:,}")

    state["complete"] = state["tokens_written"] == source["target_tokens"]
    save_snapshot(source, state, np.asarray(buffer[0], dtype=DTYPE) if buffer else np.empty(0, dtype=DTYPE))
    atomic_json(manifest_path(source), state["records"])
    assert state["complete"]
    print(f"DONE: {source['name']}")

for source in SOURCES:
    materialize_source(source)

[SKIP] FineWeb-Edu EN already complete.
[SKIP] FineWeb2-HQ PT already complete.
[SKIP] FineWeb2-HQ ES already complete.
[SKIP] OpenCoder FineWeb Code already complete.
[SKIP] OpenWebMath already complete.
[SKIP] peS2o Scientific already complete.


In [9]:

# 7. CORPUS AUDIT — FAST RESTART-SAFE VERSION
#
# Purpose:
#   Fast integrity verification for normal notebook restarts.
#
# This cell DOES NOT:
#   - recompute SHA-256 for all ~13.2 GiB of corpus data
#   - scan every token
#   - modify corpus files
#
# Full SHA-256 hashes were computed during materialization and remain
# recorded in the manifests. A deep hash audit can be run separately
# when explicitly required.

print("=" * 80)
print("MAIA CORPUS AUDIT — FAST")
print("=" * 80)

grand_total = 0
grand_shards = 0

rng = random.Random(20260929)

DTYPE_BYTES = np.dtype(DTYPE).itemsize

for source in SOURCES:

    # ------------------------------------------------------------------
    # Load authoritative materialization state
    # ------------------------------------------------------------------

    state, _carry = load_snapshot(source)

    manifest_path = (
        MANIFEST_DIR /
        f"{source['id']}.json"
    )

    assert state["complete"], (
        f"Incomplete source: {source['name']}"
    )

    assert (
        state["tokens_written"]
        == source["target_tokens"]
    ), (
        f"Token-count mismatch in snapshot: "
        f"{source['name']}"
    )

    assert manifest_path.exists(), (
        f"Missing manifest: {manifest_path}"
    )

    records = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )

    assert len(records) > 0, (
        f"Empty manifest: {source['name']}"
    )

    # ------------------------------------------------------------------
    # Fast physical-file audit
    #
    # stat() only. No complete shard reads.
    # ------------------------------------------------------------------

    physical_tokens = 0

    for record in records:

        path = (
            TRAIN_DIR /
            record["shard"]
        )

        assert path.exists(), (
            f"Missing shard: {path}"
        )

        expected_tokens = int(
            record["tokens"]
        )

        expected_bytes = (
            expected_tokens *
            DTYPE_BYTES
        )

        actual_bytes = (
            path.stat().st_size
        )

        assert (
            actual_bytes
            == expected_bytes
        ), (
            f"Shard size mismatch:\n"
            f"{path}\n"
            f"Expected: {expected_bytes:,} bytes\n"
            f"Actual:   {actual_bytes:,} bytes"
        )

        assert (
            expected_tokens
            % SEQUENCE_LENGTH
            == 0
        ), (
            f"Unaligned shard: {path}"
        )

        physical_tokens += (
            expected_tokens
        )

    assert (
        physical_tokens
        == source["target_tokens"]
    ), (
        f"Physical token mismatch: "
        f"{source['name']}"
    )

    assert (
        physical_tokens
        == state["tokens_written"]
    )

    # ------------------------------------------------------------------
    # Small semantic/token-ID spot check
    #
    # Read only 512 tokens from ONE random shard.
    # ------------------------------------------------------------------

    record = rng.choice(records)

    sample_path = (
        TRAIN_DIR /
        record["shard"]
    )

    arr = np.memmap(
        sample_path,
        dtype=DTYPE,
        mode="r",
    )

    assert len(arr) == int(
        record["tokens"]
    )

    if len(arr) > 512:

        start = rng.randrange(
            0,
            len(arr) - 512
        )

        sample = np.asarray(
            arr[
                start:
                start + 512
            ],
            dtype=np.int64,
        )

    else:

        sample = np.asarray(
            arr,
            dtype=np.int64,
        )

    assert len(sample) > 0

    assert int(sample.min()) >= 0

    assert int(sample.max()) < VOCAB_SIZE, (
        f"Invalid token ID in "
        f"{source['name']}"
    )

    eos_count = int(
        np.sum(
            sample
            == tokenizer.eos_token_id
        )
    )

    decoded_preview = tokenizer.decode(
        sample.tolist(),
        skip_special_tokens=False,
    )

    # ------------------------------------------------------------------
    # Report
    # ------------------------------------------------------------------

    grand_total += physical_tokens
    grand_shards += len(records)

    print()
    print("-" * 80)

    print(source["name"])

    print(
        f"tokens: "
        f"{physical_tokens:,}"
    )

    print(
        f"shards: "
        f"{len(records):,}"
    )

    print(
        f"sample token range: "
        f"{int(sample.min()):,} .. "
        f"{int(sample.max()):,}"
    )

    print(
        f"EOS in 512-token sample: "
        f"{eos_count}"
    )

    print(
        "sample:",
        repr(
            decoded_preview[:300]
        ),
    )

    # Explicitly release memmap.
    del arr
    del sample


# ==============================================================================
# GLOBAL CORPUS CHECK
# ==============================================================================

assert (
    grand_total
    == TARGET_TOTAL_TOKENS
), (
    f"Corpus total mismatch: "
    f"{grand_total:,} != "
    f"{TARGET_TOTAL_TOKENS:,}"
)

print()
print("=" * 80)
print("CORPUS AUDIT: PASS")
print("=" * 80)

print(
    f"Sources:      "
    f"{len(SOURCES):,}"
)

print(
    f"Shards:       "
    f"{grand_shards:,}"
)

print(
    f"Total tokens: "
    f"{grand_total:,}"
)

print(
    f"Sequences:    "
    f"{grand_total // SEQUENCE_LENGTH:,}"
)

print(
    f"Context:      "
    f"{SEQUENCE_LENGTH:,}"
)

print()
print(
    "Full SHA-256 verification: "
    "SKIPPED (restart fast path)"
)

print(
    "Manifest SHA-256 values:   "
    "PRESERVED"
)

print(
    "Corpus modification:       "
    "NONE"
)

print("=" * 80)

MAIA CORPUS AUDIT — FAST

--------------------------------------------------------------------------------
FineWeb-Edu EN
tokens: 3,193,409,536
shards: 381
sample token range: 13 .. 50,047
EOS in 512-token sample: 0
sample: ' bedrock surface.\nIn the north part of the excavation area, beside the inner face of W103, a mass of small stones (L106; Figs. 5, 6), which fills a depression (depth c. 1.5 m) in the chalk bedrock was discovered. Upon removing part of the fill, it became apparent that it predated the road, but its n'

--------------------------------------------------------------------------------
FineWeb2-HQ PT
tokens: 1,596,704,768
shards: 191
sample token range: 2 .. 48,925
EOS in 512-token sample: 2
sample: ' fato, um meta-estudo do Wooster College, em Ohio, nos Estados Unidos, analisou 250 pesquisas sobre a percepção física de homens e mulheres entre 1981 e 2012 e constatou: elas são regulamente mais insatisfeitas com a sua aparência do que eles.\nA pesquisa analisou tendênci

In [10]:

# ==============================================================================
# 8. BUILD FROZEN VALIDATION CORPUS
# ==============================================================================
#
# PURPOSE
# -------
# Build a deterministic frozen validation corpus.
#
# Validation documents are selected by SHA-256 partitioning using the SAME
# holdout rule used by the training materializer.
#
# Important properties:
#
#   - deterministic
#   - disjoint from training
#   - explicit EOS after every document
#   - no document truncation
#   - fixed token budget
#   - uint16 storage
#   - resumable at source level
#
# Existing valid validation files are preserved.
#
# ==============================================================================

import os
import time
import hashlib
from pathlib import Path

import numpy as np
from datasets import load_dataset


# ------------------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------------------

VALIDATION_TOTAL_TOKENS = 20_000 * SEQUENCE_LENGTH
# 20,480,000 tokens = 20,000 sequences of 1024 tokens


# ------------------------------------------------------------------------------
# Allocate validation tokens using the corpus mixture
# ------------------------------------------------------------------------------

def validation_target_tokens(index, source):

    total_sequences = (
        VALIDATION_TOTAL_TOKENS //
        SEQUENCE_LENGTH
    )

    if index < len(SOURCES) - 1:

        source_sequences = int(
            total_sequences *
            source["weight"]
        )

    else:

        used_sequences = sum(
            int(
                total_sequences *
                s["weight"]
            )
            for s in SOURCES[:-1]
        )

        source_sequences = (
            total_sequences -
            used_sequences
        )

    return (
        source_sequences *
        SEQUENCE_LENGTH
    )


# ------------------------------------------------------------------------------
# Fast tokenizer helper
# ------------------------------------------------------------------------------

def encode_document(text):

    ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    if not ids:
        return None

    ids.append(
        tokenizer.eos_token_id
    )

    return np.asarray(
        ids,
        dtype=DTYPE,
    )


# ------------------------------------------------------------------------------
# Build validation corpus
# ------------------------------------------------------------------------------

print("=" * 80)
print("MAIA LITE - FROZEN VALIDATION CORPUS")
print("=" * 80)

validation_records = []

for source_index, source in enumerate(SOURCES):

    target = validation_target_tokens(
        source_index,
        source,
    )

    output_file = (
        VALIDATION_DIR /
        f"{source['id']}.bin"
    )

    expected_bytes = (
        target *
        np.dtype(DTYPE).itemsize
    )

    print()
    print("-" * 80)
    print(source["name"])
    print("-" * 80)

    print(
        f"Target tokens: "
        f"{target:,}"
    )


    # --------------------------------------------------------------------------
    # Preserve already completed validation files
    # --------------------------------------------------------------------------

    if (
        output_file.exists()
        and
        output_file.stat().st_size
        ==
        expected_bytes
    ):

        print(
            "[SKIP] Valid validation file already exists"
        )

        print(
            f"File: {output_file.name}"
        )

        print(
            f"SHA-256: "
            f"{sha256_file(output_file)}"
        )

        validation_records.append(
            (
                output_file,
                target,
            )
        )

        continue


    # --------------------------------------------------------------------------
    # Remove incomplete previous attempt
    # --------------------------------------------------------------------------

    temp_file = Path(
        str(output_file) +
        ".tmp"
    )

    if temp_file.exists():
        temp_file.unlink()

    if output_file.exists():
        print(
            "[INFO] Removing incomplete validation file"
        )

        output_file.unlink()


    # --------------------------------------------------------------------------
    # Open streaming source
    # --------------------------------------------------------------------------

    print()
    print("Opening source...")

    ds = load_dataset(
        source["repo"],
        source["config"],
        split=source["split"],
        streaming=True,
    )


    # --------------------------------------------------------------------------
    # Stream validation documents directly to disk
    #
    # This avoids accumulating millions of token IDs in Python lists.
    # --------------------------------------------------------------------------

    tokens_written = 0
    documents_scanned = 0
    documents_selected = 0

    start_time = time.time()

    with open(
        temp_file,
        "wb",
        buffering=16 * 1024 * 1024,
    ) as f:

        for row in ds:

            documents_scanned += 1

            text = row.get(
                source["text_field"]
            )

            if not isinstance(
                text,
                str,
            ):
                continue

            text = text.strip()

            if not text:
                continue


            # ------------------------------------------------------------------
            # EXACT SAME deterministic validation partition used by training.
            #
            # Do NOT change this independently from the training materializer.
            # ------------------------------------------------------------------

            if not is_validation_document(
                text
            ):
                continue


            # ------------------------------------------------------------------
            # Tokenize COMPLETE document.
            #
            # No truncation.
            # Explicit EOS.
            # ------------------------------------------------------------------

            token_array = encode_document(
                text
            )

            if token_array is None:
                continue

            documents_selected += 1


            # ------------------------------------------------------------------
            # Write only what is still required.
            # ------------------------------------------------------------------

            remaining = (
                target -
                tokens_written
            )

            if len(token_array) > remaining:

                token_array = (
                    token_array[:remaining]
                )


            token_array.tofile(
                f
            )

            tokens_written += len(
                token_array
            )


            # ------------------------------------------------------------------
            # Progress
            # ------------------------------------------------------------------

            if (
                documents_selected % 1000
                ==
                0
            ):

                elapsed = (
                    time.time() -
                    start_time
                )

                rate = (
                    tokens_written /
                    max(
                        elapsed,
                        1e-9,
                    )
                )

                percent = (
                    100.0 *
                    tokens_written /
                    target
                )

                print(
                    f"selected="
                    f"{documents_selected:,}  "
                    f"scanned="
                    f"{documents_scanned:,}  "
                    f"tokens="
                    f"{tokens_written:,}/"
                    f"{target:,}  "
                    f"({percent:.1f}%)  "
                    f"rate="
                    f"{rate:,.0f} tok/s"
                )


            if tokens_written >= target:
                break


    # --------------------------------------------------------------------------
    # Validate result
    # --------------------------------------------------------------------------

    if tokens_written < target:

        if temp_file.exists():
            temp_file.unlink()

        raise RuntimeError(
            f"Not enough validation tokens "
            f"for {source['name']}: "
            f"{tokens_written:,} / "
            f"{target:,}"
        )


    actual_bytes = (
        temp_file.stat().st_size
    )

    assert (
        actual_bytes
        ==
        expected_bytes
    ), (
        f"Validation size mismatch "
        f"for {source['name']}: "
        f"{actual_bytes:,} bytes "
        f"!= "
        f"{expected_bytes:,} bytes"
    )


    # --------------------------------------------------------------------------
    # Atomic publish
    # --------------------------------------------------------------------------

    os.replace(
        temp_file,
        output_file,
    )


    # --------------------------------------------------------------------------
    # Final source report
    # --------------------------------------------------------------------------

    elapsed = (
        time.time() -
        start_time
    )

    digest = sha256_file(
        output_file
    )

    print()
    print(
        f"Saved: "
        f"{output_file.name}"
    )

    print(
        f"Tokens: "
        f"{tokens_written:,}"
    )

    print(
        f"Documents selected: "
        f"{documents_selected:,}"
    )

    print(
        f"Documents scanned: "
        f"{documents_scanned:,}"
    )

    print(
        f"Elapsed: "
        f"{elapsed / 60:.2f} min"
    )

    print(
        f"SHA-256: "
        f"{digest}"
    )


    validation_records.append(
        (
            output_file,
            target,
        )
    )


# ------------------------------------------------------------------------------
# Global validation
# ------------------------------------------------------------------------------

validation_tokens = sum(
    tokens
    for _, tokens
    in validation_records
)

validation_sequences = (
    validation_tokens //
    SEQUENCE_LENGTH
)


print()
print("=" * 80)
print("FROZEN VALIDATION CORPUS COMPLETE")
print("=" * 80)

print(
    f"Files:      "
    f"{len(validation_records):,}"
)

print(
    f"Tokens:     "
    f"{validation_tokens:,}"
)

print(
    f"Sequences:  "
    f"{validation_sequences:,}"
)

print(
    f"Context:    "
    f"{SEQUENCE_LENGTH:,}"
)

assert (
    validation_tokens
    ==
    VALIDATION_TOTAL_TOKENS
)

assert (
    validation_sequences
    ==
    20_000
)

print()
print(
    "VALIDATION CORPUS: PASS"
)

MAIA LITE - FROZEN VALIDATION CORPUS

--------------------------------------------------------------------------------
FineWeb-Edu EN
--------------------------------------------------------------------------------
Target tokens: 9,216,000
[SKIP] Valid validation file already exists
File: fineweb_edu_en.bin
SHA-256: 9599f4fbc8ff89421e8a38ebcd125a0e1d8746732a79eb6de4f28e192280b08d

--------------------------------------------------------------------------------
FineWeb2-HQ PT
--------------------------------------------------------------------------------
Target tokens: 4,608,000
[SKIP] Valid validation file already exists
File: fineweb2_hq_pt.bin
SHA-256: 5807ef40ae6b6087164484afcc45da32cda3076246d70cc0a1a4115ceed4c6e5

--------------------------------------------------------------------------------
FineWeb2-HQ ES
--------------------------------------------------------------------------------
Target tokens: 4,608,000
[SKIP] Valid validation file already exists
File: fineweb2_hq_es.bin

## Model initialization

The model is initialized **from scratch** with the GPT-2 Medium architecture. The parameter-count assertion is mandatory: training must not start unless it is exactly 354,823,168 parameters.


In [11]:
# 8. INITIALIZE EXACT GPT-2 MEDIUM-MATCHED MAIA LITE

config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=SEQUENCE_LENGTH,
    n_ctx=SEQUENCE_LENGTH,
    n_embd=1024,
    n_layer=24,
    n_head=16,
    n_inner=4096,
    activation_function="gelu_new",
    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
    layer_norm_epsilon=1e-5,
    initializer_range=0.02,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    use_cache=True,
)

model = GPT2LMHeadModel(config)
parameter_count = sum(p.numel() for p in model.parameters())

print(f"Parameters: {parameter_count:,}")
print(config)

assert parameter_count == N_PARAMS_REFERENCE, (
    f"Parameter mismatch: {parameter_count:,} != {N_PARAMS_REFERENCE:,}"
)

print("\nExact GPT-2 Medium parameter match: PASS")

Parameters: 354,823,168
GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 2,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1024,
  "n_head": 16,
  "n_inner": 4096,
  "n_layer": 24,
  "n_positions": 1024,
  "pad_token_id": 1,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0",
  "use_cache": true,
  "vocab_size": 50257
}


Exact GPT-2 Medium parameter match: PASS


## Training

Training is deliberately separated from corpus construction. Before running this section:
- corpus audit must pass;
- frozen validation must be created;
- random decoded samples must be inspected manually.

Recommended behavioral gates: 1k, 5k, 10k, 25k, 50k, 100k, then regular checkpoints through 216,567.


In [ ]:

# ==============================================================================
# 9. PACK CORPUS ONCE + STAGE SINGLE BINARIES TO LOCAL SSD
# ==============================================================================
#
# ARCHITECTURE
#
# Google Drive (persistent):
#
#   corpus/train/*.bin              <- original 849 canonical shards
#   corpus/manifests/*.json         <- original manifests
#
#   corpus/packed/train.bin         <- created ONCE
#   corpus/packed/train.bin.sha256  <- created ONCE
#
#   corpus/packed/validation.bin
#   corpus/packed/validation.bin.sha256
#
#
# Local Colab SSD (every runtime):
#
#   /content/maia_lite_v3/train.bin
#   /content/maia_lite_v3/validation.bin
#
#
# Trainer reads ONLY the local SSD files.
#
# ==============================================================================

import os
import json
import time
import shutil
import hashlib
from pathlib import Path

import numpy as np
import torch

from torch.utils.data import Dataset


# ==============================================================================
# CONFIGURATION
# ==============================================================================

LOCAL_ROOT = Path(
    "/content/maia_lite_v3"
)

LOCAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


PACKED_DIR = (
    CORPUS_DIR /
    "packed"
)

PACKED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DRIVE_TRAIN_BIN = (
    PACKED_DIR /
    "train.bin"
)

DRIVE_TRAIN_SHA = (
    PACKED_DIR /
    "train.bin.sha256"
)


DRIVE_VALIDATION_BIN = (
    PACKED_DIR /
    "validation.bin"
)

DRIVE_VALIDATION_SHA = (
    PACKED_DIR /
    "validation.bin.sha256"
)


LOCAL_TRAIN_BIN = (
    LOCAL_ROOT /
    "train.bin"
)

LOCAL_VALIDATION_BIN = (
    LOCAL_ROOT /
    "validation.bin"
)


DTYPE_BYTES = np.dtype(
    DTYPE
).itemsize


# ==============================================================================
# SHA-256
# ==============================================================================

def sha256_stream(
    path,
    chunk_size=16 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
        buffering=0,
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# ==============================================================================
# BUILD CANONICAL TRAINING INVENTORY
# ==============================================================================

canonical_records = []

for source in SOURCES:

    manifest_path = (
        MANIFEST_DIR /
        f"{source['id']}.json"
    )

    assert manifest_path.exists(), (
        f"Missing manifest: "
        f"{manifest_path}"
    )

    manifest = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )

    for record in manifest:

        canonical_path = (
            TRAIN_DIR /
            record["shard"]
        )

        tokens = int(
            record["tokens"]
        )

        assert canonical_path.exists(), (
            f"Missing canonical shard: "
            f"{canonical_path}"
        )

        expected_bytes = (
            tokens *
            DTYPE_BYTES
        )

        actual_bytes = (
            canonical_path
            .stat()
            .st_size
        )

        assert (
            actual_bytes
            ==
            expected_bytes
        ), (
            f"Canonical shard size mismatch:\n"
            f"{canonical_path}\n"
            f"Expected: {expected_bytes:,}\n"
            f"Actual:   {actual_bytes:,}"
        )

        canonical_records.append(
            (
                canonical_path,
                tokens,
            )
        )


canonical_tokens = sum(
    tokens
    for _, tokens
    in canonical_records
)


canonical_bytes = (
    canonical_tokens *
    DTYPE_BYTES
)


assert (
    canonical_tokens
    ==
    TARGET_TOTAL_TOKENS
)


print("=" * 80)
print("MAIA LITE - PACKED CORPUS")
print("=" * 80)

print()

print(
    f"Canonical shards: "
    f"{len(canonical_records):,}"
)

print(
    f"Training tokens:  "
    f"{canonical_tokens:,}"
)

print(
    f"Training size:    "
    f"{canonical_bytes / 1024**3:.2f} GiB"
)


# ==============================================================================
# PACK TRAINING CORPUS — ONLY IF NECESSARY
# ==============================================================================

train_packed_valid = (
    DRIVE_TRAIN_BIN.exists()
    and
    DRIVE_TRAIN_SHA.exists()
    and
    DRIVE_TRAIN_BIN.stat().st_size
    ==
    canonical_bytes
)


if train_packed_valid:

    print()
    print(
        "Persistent train.bin: FOUND"
    )

    print(
        "Size check:           PASS"
    )

    print(
        "Repacking shards:     NO"
    )


else:

    print()
    print("=" * 80)
    print("CREATING PERSISTENT train.bin")
    print("=" * 80)

    print(
        "This happens only once."
    )

    print(
        "Concatenating canonical shards..."
    )


    temp_train = Path(
        str(DRIVE_TRAIN_BIN) +
        ".tmp"
    )


    if temp_train.exists():
        temp_train.unlink()


    digest = hashlib.sha256()

    bytes_written = 0

    start_time = time.time()


    with open(
        temp_train,
        "wb",
        buffering=16 * 1024 * 1024,
    ) as dst:

        for index, (
            shard_path,
            tokens,
        ) in enumerate(
            canonical_records,
            start=1,
        ):

            with open(
                shard_path,
                "rb",
                buffering=16 * 1024 * 1024,
            ) as src:

                while True:

                    chunk = src.read(
                        16 * 1024 * 1024
                    )

                    if not chunk:
                        break

                    dst.write(
                        chunk
                    )

                    digest.update(
                        chunk
                    )

                    bytes_written += (
                        len(chunk)
                    )


            if (
                index == 1
                or
                index % 50 == 0
                or
                index == len(
                    canonical_records
                )
            ):

                elapsed = (
                    time.time()
                    -
                    start_time
                )

                print(
                    f"{index:4d}/"
                    f"{len(canonical_records):4d}  "
                    f"{bytes_written / 1024**3:.2f} GiB  "
                    f"{elapsed / 60:.1f} min"
                )


        dst.flush()

        os.fsync(
            dst.fileno()
        )


    assert (
        bytes_written
        ==
        canonical_bytes
    )


    assert (
        temp_train.stat().st_size
        ==
        canonical_bytes
    )


    train_hash = (
        digest.hexdigest()
    )


    # Atomic publication of train.bin

    os.replace(
        temp_train,
        DRIVE_TRAIN_BIN,
    )


    # Atomic publication of hash

    temp_sha = Path(
        str(DRIVE_TRAIN_SHA) +
        ".tmp"
    )

    temp_sha.write_text(
        train_hash + "\n",
        encoding="ascii",
    )

    os.replace(
        temp_sha,
        DRIVE_TRAIN_SHA,
    )


    print()
    print(
        "Persistent train.bin created."
    )

    print(
        f"SHA-256: {train_hash}"
    )


# ==============================================================================
# LOAD AUTHORITATIVE TRAIN HASH
# ==============================================================================

assert DRIVE_TRAIN_BIN.exists()

assert DRIVE_TRAIN_SHA.exists()

assert (
    DRIVE_TRAIN_BIN.stat().st_size
    ==
    canonical_bytes
)


expected_train_hash = (
    DRIVE_TRAIN_SHA
    .read_text(
        encoding="ascii"
    )
    .strip()
)


assert (
    len(expected_train_hash)
    ==
    64
)


print()
print(
    f"Persistent train.bin: "
    f"{DRIVE_TRAIN_BIN}"
)

print(
    f"Expected SHA-256:      "
    f"{expected_train_hash}"
)


# ==============================================================================
# BUILD VALIDATION INVENTORY
# ==============================================================================

validation_tokens = sum(
    int(tokens)
    for _, tokens
    in validation_records
)


validation_bytes = (
    validation_tokens *
    DTYPE_BYTES
)


assert (
    validation_tokens
    ==
    20_480_000
)


# ==============================================================================
# PACK VALIDATION — ONLY IF NECESSARY
# ==============================================================================

validation_packed_valid = (
    DRIVE_VALIDATION_BIN.exists()
    and
    DRIVE_VALIDATION_SHA.exists()
    and
    DRIVE_VALIDATION_BIN.stat().st_size
    ==
    validation_bytes
)


if validation_packed_valid:

    print()
    print(
        "Persistent validation.bin: FOUND"
    )

    print(
        "Validation repacking:       NO"
    )


else:

    print()
    print("=" * 80)
    print("CREATING PERSISTENT validation.bin")
    print("=" * 80)


    temp_validation = Path(
        str(DRIVE_VALIDATION_BIN) +
        ".tmp"
    )


    if temp_validation.exists():
        temp_validation.unlink()


    digest = hashlib.sha256()

    bytes_written = 0


    with open(
        temp_validation,
        "wb",
        buffering=16 * 1024 * 1024,
    ) as dst:

        for (
            canonical_path,
            tokens,
        ) in validation_records:

            canonical_path = Path(
                canonical_path
            )

            expected_bytes = (
                int(tokens) *
                DTYPE_BYTES
            )

            assert (
                canonical_path.exists()
            )

            assert (
                canonical_path
                .stat()
                .st_size
                ==
                expected_bytes
            )


            with open(
                canonical_path,
                "rb",
                buffering=16 * 1024 * 1024,
            ) as src:

                while True:

                    chunk = src.read(
                        16 * 1024 * 1024
                    )

                    if not chunk:
                        break

                    dst.write(
                        chunk
                    )

                    digest.update(
                        chunk
                    )

                    bytes_written += (
                        len(chunk)
                    )


        dst.flush()

        os.fsync(
            dst.fileno()
        )


    assert (
        bytes_written
        ==
        validation_bytes
    )

    assert (
        temp_validation.stat().st_size
        ==
        validation_bytes
    )


    validation_hash = (
        digest.hexdigest()
    )


    os.replace(
        temp_validation,
        DRIVE_VALIDATION_BIN,
    )


    temp_sha = Path(
        str(DRIVE_VALIDATION_SHA) +
        ".tmp"
    )

    temp_sha.write_text(
        validation_hash + "\n",
        encoding="ascii",
    )

    os.replace(
        temp_sha,
        DRIVE_VALIDATION_SHA,
    )


    print(
        "Persistent validation.bin created."
    )

    print(
        f"SHA-256: {validation_hash}"
    )


# ==============================================================================
# LOAD VALIDATION HASH
# ==============================================================================

assert DRIVE_VALIDATION_BIN.exists()

assert DRIVE_VALIDATION_SHA.exists()

assert (
    DRIVE_VALIDATION_BIN.stat().st_size
    ==
    validation_bytes
)


expected_validation_hash = (
    DRIVE_VALIDATION_SHA
    .read_text(
        encoding="ascii"
    )
    .strip()
)


assert (
    len(expected_validation_hash)
    ==
    64
)


# ==============================================================================
# CHECK LOCAL SSD CAPACITY
# ==============================================================================

disk = shutil.disk_usage(
    "/content"
)


additional_required = 0


if (
    not LOCAL_TRAIN_BIN.exists()
    or
    LOCAL_TRAIN_BIN.stat().st_size
    !=
    canonical_bytes
):

    additional_required += (
        canonical_bytes
    )


if (
    not LOCAL_VALIDATION_BIN.exists()
    or
    LOCAL_VALIDATION_BIN.stat().st_size
    !=
    validation_bytes
):

    additional_required += (
        validation_bytes
    )


print()
print("=" * 80)
print("LOCAL SSD")
print("=" * 80)

print(
    f"Free:       "
    f"{disk.free / 1024**3:.2f} GiB"
)

print(
    f"Need copy:  "
    f"{additional_required / 1024**3:.2f} GiB"
)


assert (
    disk.free
    >=
    additional_required
), (
    "Not enough local SSD space."
)


# ==============================================================================
# COPY + VERIFY ONE PERSISTENT BINARY
# ==============================================================================

def stage_binary(
    source,
    destination,
    expected_bytes,
    expected_hash,
    label,
):

    print()
    print("=" * 80)
    print(
        f"STAGING {label}"
    )
    print("=" * 80)


    local_valid = (
        destination.exists()
        and
        destination.stat().st_size
        ==
        expected_bytes
    )


    if local_valid:

        print(
            "Local file with correct "
            "size already exists."
        )

        print(
            "Verifying local SHA-256..."
        )


        actual_hash = (
            sha256_stream(
                destination
            )
        )


        if (
            actual_hash
            ==
            expected_hash
        ):

            print(
                "Local SHA-256: PASS"
            )

            print(
                "Copy required: NO"
            )

            return


        print(
            "Local SHA-256 mismatch."
        )

        destination.unlink()


    elif destination.exists():

        destination.unlink()


    temp_path = Path(
        str(destination) +
        ".tmp"
    )


    if temp_path.exists():
        temp_path.unlink()


    print(
        f"Copying:\n"
        f"  {source}\n"
        f"→ {destination}"
    )


    start_time = time.time()

    copied_bytes = 0


    digest = hashlib.sha256()


    with open(
        source,
        "rb",
        buffering=16 * 1024 * 1024,
    ) as src, open(
        temp_path,
        "wb",
        buffering=16 * 1024 * 1024,
    ) as dst:

        while True:

            chunk = src.read(
                16 * 1024 * 1024
            )

            if not chunk:
                break

            dst.write(
                chunk
            )

            digest.update(
                chunk
            )

            copied_bytes += (
                len(chunk)
            )


        dst.flush()

        os.fsync(
            dst.fileno()
        )


    elapsed = (
        time.time()
        -
        start_time
    )


    assert (
        copied_bytes
        ==
        expected_bytes
    )


    actual_hash = (
        digest.hexdigest()
    )


    assert (
        actual_hash
        ==
        expected_hash
    ), (
        f"{label} SHA-256 mismatch "
        f"after copy."
    )


    os.replace(
        temp_path,
        destination,
    )


    print(
        f"Copied: "
        f"{copied_bytes / 1024**3:.2f} GiB"
    )

    print(
        f"Elapsed: "
        f"{elapsed / 60:.1f} min"
    )

    print(
        "SHA-256: PASS"
    )


# ==============================================================================
# STAGE TRAINING BIN
# ==============================================================================

stage_binary(
    source=DRIVE_TRAIN_BIN,
    destination=LOCAL_TRAIN_BIN,
    expected_bytes=canonical_bytes,
    expected_hash=expected_train_hash,
    label="TRAINING CORPUS",
)


# ==============================================================================
# STAGE VALIDATION BIN
# ==============================================================================

stage_binary(
    source=DRIVE_VALIDATION_BIN,
    destination=LOCAL_VALIDATION_BIN,
    expected_bytes=validation_bytes,
    expected_hash=expected_validation_hash,
    label="VALIDATION CORPUS",
)


# ==============================================================================
# SINGLE-FILE MAIA BINARY DATASET
# ==============================================================================

class MaiaBinaryDataset(Dataset):

    def __init__(
        self,
        path,
        total_tokens,
        sequence_length=1024,
    ):

        self.path = Path(
            path
        )

        self.total_tokens = int(
            total_tokens
        )

        self.sequence_length = int(
            sequence_length
        )


        assert (
            self.total_tokens
            %
            self.sequence_length
            ==
            0
        )


        expected_bytes = (
            self.total_tokens *
            DTYPE_BYTES
        )


        assert (
            self.path.exists()
        )


        assert (
            self.path.stat().st_size
            ==
            expected_bytes
        )


        self.total_sequences = (
            self.total_tokens
            //
            self.sequence_length
        )


        self._map = None


    def __len__(
        self
    ):

        return (
            self.total_sequences
        )


    def _memmap(
        self
    ):

        if self._map is None:

            self._map = np.memmap(
                self.path,
                dtype=DTYPE,
                mode="r",
            )

        return (
            self._map
        )


    def __getitem__(
        self,
        index,
    ):

        if index < 0:

            index += (
                self.total_sequences
            )


        if (
            index < 0
            or
            index >=
            self.total_sequences
        ):

            raise IndexError(
                index
            )


        arr = self._memmap()


        start = (
            index *
            self.sequence_length
        )

        end = (
            start +
            self.sequence_length
        )


        ids_np = np.asarray(
            arr[
                start:end
            ],
            dtype=np.int64,
        )


        assert (
            len(ids_np)
            ==
            self.sequence_length
        )


        ids = torch.from_numpy(
            ids_np
        ).long()


        return {
            "input_ids": ids,

            "attention_mask":
                torch.ones_like(
                    ids
                ),

            "labels":
                ids.clone(),
        }


# ==============================================================================
# BUILD TRAINING DATASET
# ==============================================================================

train_dataset = MaiaBinaryDataset(
    path=LOCAL_TRAIN_BIN,
    total_tokens=TARGET_TOTAL_TOKENS,
    sequence_length=SEQUENCE_LENGTH,
)


# ==============================================================================
# BUILD VALIDATION DATASET
# ==============================================================================

eval_dataset = MaiaBinaryDataset(
    path=LOCAL_VALIDATION_BIN,
    total_tokens=validation_tokens,
    sequence_length=SEQUENCE_LENGTH,
)


# ==============================================================================
# FINAL ASSERTIONS
# ==============================================================================

training_sequences = len(
    train_dataset
)

training_tokens = (
    training_sequences *
    SEQUENCE_LENGTH
)


assert (
    training_tokens
    ==
    TARGET_TOTAL_TOKENS
)


assert (
    len(eval_dataset)
    ==
    20_000
)


assert str(
    train_dataset.path
).startswith(
    "/content/"
)


assert str(
    eval_dataset.path
).startswith(
    "/content/"
)


# ==============================================================================
# READ TEST
# ==============================================================================

test_item = (
    train_dataset[0]
)


assert (
    test_item[
        "input_ids"
    ].shape[0]
    ==
    SEQUENCE_LENGTH
)


assert (
    int(
        test_item[
            "input_ids"
        ].max()
    )
    <
    VOCAB_SIZE
)


assert (
    int(
        test_item[
            "input_ids"
        ].min()
    )
    >=
    0
)


eval_test_item = (
    eval_dataset[0]
)


assert (
    eval_test_item[
        "input_ids"
    ].shape[0]
    ==
    SEQUENCE_LENGTH
)


# ==============================================================================
# REPORT
# ==============================================================================

print()
print("=" * 80)
print("MAIA LITE - LOCAL DATASETS READY")
print("=" * 80)

print(
    f"Training file:      "
    f"{LOCAL_TRAIN_BIN}"
)

print(
    f"Training sequences: "
    f"{len(train_dataset):,}"
)

print(
    f"Training tokens:    "
    f"{TARGET_TOTAL_TOKENS:,}"
)

print(
    f"Training size:      "
    f"{canonical_bytes / 1024**3:.2f} GiB"
)

print()

print(
    f"Validation file:      "
    f"{LOCAL_VALIDATION_BIN}"
)

print(
    f"Validation sequences: "
    f"{len(eval_dataset):,}"
)

print(
    f"Validation tokens:    "
    f"{validation_tokens:,}"
)

print()

print(
    "Training I/O:       LOCAL SSD"
)

print(
    "Validation I/O:     LOCAL SSD"
)

print(
    "Persistent corpus:  SINGLE PACKED BINARIES"
)

print(
    "SHA-256 validation: PASS"
)

print(
    "DATASET STAGING:    PASS"
)

print("=" * 80)

MAIA LITE - PACKED CORPUS

Canonical shards: 849
Training tokens:  7,096,467,456
Training size:    13.22 GiB

Persistent train.bin: FOUND
Size check:           PASS
Repacking shards:     NO

Persistent train.bin: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/corpus/packed/train.bin
Expected SHA-256:      a8176dfca40f508f2fe6fd19512fcd2e78b76ab0dd4e589030f6134d0df39338

Persistent validation.bin: FOUND
Validation repacking:       NO

LOCAL SSD
Free:       192.14 GiB
Need copy:  13.26 GiB

STAGING TRAINING CORPUS
Copying:
  /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/corpus/packed/train.bin
→ /content/maia_lite_v3/train.bin
Copied: 13.22 GiB
Elapsed: 10.6 min
SHA-256: PASS

STAGING VALIDATION CORPUS
Copying:
  /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/corpus/packed/validation.bin
→ /content/maia_lite_v3/validation.bin
Copied: 0.04 GiB
Elapsed: 0.1 min
SHA-256: PASS

MAIA LITE - LOCAL DATASETS READY
Training file:      /content/maia_lite_v3/train.bin
Training sequences: 6,930,144

In [ ]:

# ==============================================================================
# MAIA LITE - FINAL PREFLIGHT
# ==============================================================================
#
# Final read-only validation before switching to the A100 training runtime.
#
# Expected architecture:
#
#   Persistent:
#       Google Drive / corpus / packed / train.bin
#       Google Drive / corpus / packed / validation.bin
#
#   Runtime:
#       /content/maia_lite_v3/train.bin
#       /content/maia_lite_v3/validation.bin
#
# Trainer must read ONLY from local SSD.
#
# ==============================================================================

import os
import numpy as np
import torch


print("=" * 80)
print("MAIA LITE - FINAL PREFLIGHT")
print("=" * 80)


# ==============================================================================
# 1. TOKENIZER
# ==============================================================================

assert tokenizer is not None

assert len(tokenizer) == VOCAB_SIZE

assert VOCAB_SIZE == 50_257

assert tokenizer.bos_token_id == 0
assert tokenizer.pad_token_id == 1
assert tokenizer.eos_token_id == 2
assert tokenizer.unk_token_id == 3
assert tokenizer.mask_token_id == 4


# Known tokenizer identity regression tests.

for text, expected_ids in KNOWN_TOKENIZATION.items():

    actual_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    assert (
        actual_ids
        ==
        expected_ids
    ), (
        f"Tokenizer identity failure:\n"
        f"{text!r}\n"
        f"Expected: {expected_ids}\n"
        f"Actual:   {actual_ids}"
    )


print("Tokenizer:           PASS")


# ==============================================================================
# 2. TRAINING BUDGET
# ==============================================================================

assert N_PARAMS_REFERENCE == 354_823_168

assert TARGET_STEPS == 216_567

assert EFFECTIVE_BATCH == 32

assert SEQUENCE_LENGTH == 1024


tokens_per_step = (
    EFFECTIVE_BATCH *
    SEQUENCE_LENGTH
)


assert (
    tokens_per_step
    ==
    32_768
)


calculated_total_tokens = (
    TARGET_STEPS *
    tokens_per_step
)


assert (
    calculated_total_tokens
    ==
    TARGET_TOTAL_TOKENS
)


assert (
    TARGET_TOTAL_TOKENS
    ==
    7_096_467_456
)


tokens_per_parameter = (
    TARGET_TOTAL_TOKENS /
    N_PARAMS_REFERENCE
)


print("Training budget:     PASS")


# ==============================================================================
# 3. TRAINING DATASET
# ==============================================================================

assert hasattr(
    train_dataset,
    "path",
)

assert hasattr(
    train_dataset,
    "total_tokens",
)

assert hasattr(
    train_dataset,
    "total_sequences",
)


train_path = Path(
    train_dataset.path
)


assert train_path.exists()


assert str(
    train_path
).startswith(
    "/content/"
), (
    "Training dataset is not on local SSD."
)


assert (
    train_path.resolve()
    ==
    LOCAL_TRAIN_BIN.resolve()
)


assert (
    train_dataset.total_tokens
    ==
    TARGET_TOTAL_TOKENS
)


assert (
    len(train_dataset)
    ==
    6_930_144
)


assert (
    len(train_dataset)
    ==
    TARGET_STEPS *
    EFFECTIVE_BATCH
)


expected_train_bytes = (
    TARGET_TOTAL_TOKENS *
    np.dtype(DTYPE).itemsize
)


assert (
    train_path.stat().st_size
    ==
    expected_train_bytes
)


print("Training I/O:        LOCAL SSD - PASS")


# ==============================================================================
# 4. VALIDATION DATASET
# ==============================================================================

assert hasattr(
    eval_dataset,
    "path",
)

assert hasattr(
    eval_dataset,
    "total_tokens",
)


eval_path = Path(
    eval_dataset.path
)


assert eval_path.exists()


assert str(
    eval_path
).startswith(
    "/content/"
), (
    "Validation dataset is not on local SSD."
)


assert (
    eval_path.resolve()
    ==
    LOCAL_VALIDATION_BIN.resolve()
)


assert (
    eval_dataset.total_tokens
    ==
    20_480_000
)


assert (
    len(eval_dataset)
    ==
    20_000
)


expected_validation_bytes = (
    20_480_000 *
    np.dtype(DTYPE).itemsize
)


assert (
    eval_path.stat().st_size
    ==
    expected_validation_bytes
)


print("Validation corpus:   PASS")
print("Validation I/O:      LOCAL SSD - PASS")


# ==============================================================================
# 5. TRAINING DATA READ TEST
# ==============================================================================

train_first = train_dataset[0]

train_middle = train_dataset[
    len(train_dataset) // 2
]

train_last = train_dataset[
    len(train_dataset) - 1
]


for sample in [
    train_first,
    train_middle,
    train_last,
]:

    ids = sample[
        "input_ids"
    ]

    labels = sample[
        "labels"
    ]

    attention = sample[
        "attention_mask"
    ]


    assert (
        ids.shape[0]
        ==
        SEQUENCE_LENGTH
    )

    assert (
        labels.shape
        ==
        ids.shape
    )

    assert (
        attention.shape
        ==
        ids.shape
    )

    assert torch.equal(
        ids,
        labels,
    )

    assert int(
        ids.min()
    ) >= 0

    assert int(
        ids.max()
    ) < VOCAB_SIZE


print("Training read test:  PASS")


# ==============================================================================
# 6. VALIDATION DATA READ TEST
# ==============================================================================

eval_first = eval_dataset[0]

eval_last = eval_dataset[
    len(eval_dataset) - 1
]


for sample in [
    eval_first,
    eval_last,
]:

    ids = sample[
        "input_ids"
    ]

    assert (
        ids.shape[0]
        ==
        SEQUENCE_LENGTH
    )

    assert int(
        ids.min()
    ) >= 0

    assert int(
        ids.max()
    ) < VOCAB_SIZE


print("Validation read:     PASS")


# ==============================================================================
# 7. MODEL ARCHITECTURE
# ==============================================================================

actual_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)


assert (
    actual_parameters
    ==
    N_PARAMS_REFERENCE
), (
    f"Parameter mismatch: "
    f"{actual_parameters:,} != "
    f"{REFERENCE_PARAM_COUNT:,}"
)


assert (
    model.config.vocab_size
    ==
    VOCAB_SIZE
)

assert (
    model.config.n_positions
    ==
    SEQUENCE_LENGTH
)

assert (
    model.config.n_ctx
    ==
    SEQUENCE_LENGTH
)

assert (
    model.config.n_embd
    ==
    1024
)

assert (
    model.config.n_layer
    ==
    24
)

assert (
    model.config.n_head
    ==
    16
)


print("Model architecture:  PASS")


# ==============================================================================
# 8. PACKED PERSISTENT CORPUS
# ==============================================================================

assert DRIVE_TRAIN_BIN.exists()

assert DRIVE_TRAIN_SHA.exists()

assert DRIVE_VALIDATION_BIN.exists()

assert DRIVE_VALIDATION_SHA.exists()


assert (
    DRIVE_TRAIN_BIN.stat().st_size
    ==
    expected_train_bytes
)


assert (
    DRIVE_VALIDATION_BIN.stat().st_size
    ==
    expected_validation_bytes
)


persistent_train_hash = (
    DRIVE_TRAIN_SHA
    .read_text(
        encoding="ascii"
    )
    .strip()
)


persistent_validation_hash = (
    DRIVE_VALIDATION_SHA
    .read_text(
        encoding="ascii"
    )
    .strip()
)


assert (
    len(persistent_train_hash)
    ==
    64
)

assert (
    len(persistent_validation_hash)
    ==
    64
)


print("Packed corpus:       PASS")


# ==============================================================================
# FINAL REPORT
# ==============================================================================

print()
print("=" * 80)
print("MAIA LITE - DEFINITIVE PRETRAINING CONFIGURATION")
print("=" * 80)

print(
    f"Parameters:           "
    f"{actual_parameters:,}"
)

print(
    f"Vocabulary:           "
    f"{VOCAB_SIZE:,}"
)

print(
    f"Context length:       "
    f"{SEQUENCE_LENGTH:,}"
)

print()

print(
    f"Training sequences:   "
    f"{len(train_dataset):,}"
)

print(
    f"Training tokens:      "
    f"{TARGET_TOTAL_TOKENS:,}"
)

print(
    f"Validation sequences: "
    f"{len(eval_dataset):,}"
)

print(
    f"Validation tokens:    "
    f"{eval_dataset.total_tokens:,}"
)

print()

print(
    f"Micro batch:          "
    f"{MICRO_BATCH}"
)

print(
    f"Gradient accumulation:"
    f" {GRAD_ACCUM}"
)

print(
    f"Effective batch:      "
    f"{EFFECTIVE_BATCH}"
)

print(
    f"Tokens / step:        "
    f"{tokens_per_step:,}"
)

print(
    f"Optimizer steps:      "
    f"{TARGET_STEPS:,}"
)

print(
    f"Tokens / parameter:   "
    f"{tokens_per_parameter:.6f}"
)

print()

print(
    f"Training path:        "
    f"{train_path}"
)

print(
    f"Validation path:      "
    f"{eval_path}"
)

print()

print(
    f"Train SHA-256:        "
    f"{persistent_train_hash}"
)

print(
    f"Validation SHA-256:   "
    f"{persistent_validation_hash}"
)

print()

print(
    "Training storage:     LOCAL SSD"
)

print(
    "Validation storage:   LOCAL SSD"
)

print(
    "Persistent storage:   GOOGLE DRIVE"
)

print(
    "Original shards:      PRESERVED"
)

print()

print("=" * 80)
print("FINAL PREFLIGHT: PASS")
print("=" * 80)

MAIA LITE - FINAL PREFLIGHT
Tokenizer:           PASS
Training budget:     PASS
Training I/O:        LOCAL SSD - PASS
Validation corpus:   PASS
Validation I/O:      LOCAL SSD - PASS
Training read test:  PASS
Validation read:     PASS
Model architecture:  PASS
Packed corpus:       PASS

MAIA LITE - DEFINITIVE PRETRAINING CONFIGURATION
Parameters:           354,823,168
Vocabulary:           50,257
Context length:       1,024

Training sequences:   6,930,144
Training tokens:      7,096,467,456
Validation sequences: 20,000
Validation tokens:    20,480,000

Micro batch:          16
Gradient accumulation: 2
Effective batch:      32
Tokens / step:        32,768
Optimizer steps:      216,567
Tokens / parameter:   20.000012

Training path:        /content/maia_lite_v3/train.bin
Validation path:      /content/maia_lite_v3/validation.bin

Train SHA-256:        a8176dfca40f508f2fe6fd19512fcd2e78b76ab0dd4e589030f6134d0df39338
Validation SHA-256:   43b5584117fc52dd5e4e1068536e4acf15dd606d06c571f0390

In [ ]:
# ==============================================================================
# 10. A100 TRAINING CONFIGURATION
# ==============================================================================
#
# Run this cell ONLY in the A100 training runtime.
# CPU preparation/restart validation ends at the Final Preflight above.
#
# Full recovery checkpoints are kept on Google Drive with a rolling retention
# window. Scientific milestone snapshots should be exported separately.
# ==============================================================================

import os
import torch
import transformers
from transformers import TrainingArguments

assert torch.cuda.is_available(), (
    "Cell 10 is A100-only. Final Preflight can be run on CPU; "
    "create TrainingArguments only in the GPU training runtime."
)

gpu_name = torch.cuda.get_device_name(0)

assert "A100" in gpu_name.upper(), (
    f"Expected NVIDIA A100 for definitive training, got: {gpu_name}"
)

assert torch.cuda.is_bf16_supported(), (
    "A100 runtime does not report BF16 support."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

assert MICRO_BATCH * GRAD_ACCUM == EFFECTIVE_BATCH
assert TARGET_STEPS * EFFECTIVE_BATCH * SEQUENCE_LENGTH == TARGET_TOTAL_TOKENS
assert len(train_dataset) == TARGET_STEPS * EFFECTIVE_BATCH
assert len(eval_dataset) == 20_000

LEARNING_RATE = 3e-4
WARMUP_STEPS = 2_000
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

LOGGING_STEPS = 25
SAVE_STEPS = 1_000
SAVE_TOTAL_LIMIT = 3

# KV cache is unnecessary during gradient training and wastes memory.
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=MICRO_BATCH,

    max_steps=TARGET_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,

    bf16=True,
    fp16=False,
    tf32=True,

    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    # Validation is frozen and attached to Trainer, but expensive full-corpus
    # evaluation is run deliberately at scientific checkpoints, not every 1k.
    eval_strategy="no",
    prediction_loss_only=True,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    save_only_model=False,

    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    dataloader_drop_last=True,

    seed=SEED,
    data_seed=SEED,

    report_to="none",
    remove_unused_columns=False,

    # Critical for deterministic full-state resume.
    ignore_data_skip=False,
)

print("=" * 80)
print("MAIA LITE - A100 TRAINING CONFIGURATION")
print("=" * 80)
print(f"Transformers:          {transformers.__version__}")
print(f"PyTorch:               {torch.__version__}")
print(f"GPU:                   {gpu_name}")
print(f"BF16 supported:        {torch.cuda.is_bf16_supported()}")
print(f"Micro batch:           {MICRO_BATCH}")
print(f"Gradient accumulation: {GRAD_ACCUM}")
print(f"Effective batch:       {EFFECTIVE_BATCH}")
print(f"Tokens / step:         {EFFECTIVE_BATCH * SEQUENCE_LENGTH:,}")
print(f"Optimizer steps:       {TARGET_STEPS:,}")
print(f"Total tokens:          {TARGET_TOTAL_TOKENS:,}")
print(f"Learning rate:         {LEARNING_RATE:.2e}")
print(f"Warmup steps:          {WARMUP_STEPS:,}")
print(f"Checkpoint every:      {SAVE_STEPS:,} steps")
print(f"Rolling checkpoints:   {SAVE_TOTAL_LIMIT}")
print("Validation strategy:   manual frozen-corpus evaluation")
print("Training I/O:          LOCAL SSD")
print("Validation I/O:        LOCAL SSD")
print("Checkpoints:           GOOGLE DRIVE")
print("A100 TRAINING CONFIGURATION: PASS")
print("=" * 80)


MAIA LITE - A100 TRAINING CONFIGURATION
Transformers:          5.17.0
PyTorch:               2.11.0+cu130
GPU:                   NVIDIA A100-SXM4-80GB
BF16 supported:        True
Micro batch:           16
Gradient accumulation: 2
Effective batch:       32
Tokens / step:         32,768
Optimizer steps:       216,567
Total tokens:          7,096,467,456
Learning rate:         3.00e-04
Warmup steps:          2,000
Checkpoint every:      1,000 steps
Rolling checkpoints:   3
Validation strategy:   manual frozen-corpus evaluation
Training I/O:          LOCAL SSD
Validation I/O:        LOCAL SSD
Checkpoints:           GOOGLE DRIVE
A100 TRAINING CONFIGURATION: PASS


In [ ]:

# ==============================================================================
# 11. BUILD TRAINER + DETERMINISTIC GLOBAL SHUFFLE + RESUME AUDIT
# ==============================================================================
#
# This cell DOES NOT start training.
#
# Requirements:
#   - Global randomization across the complete packed training corpus.
#   - Deterministic ordering from DATA_SEED.
#   - Full checkpoint detection for exact Trainer resume.
#   - Frozen validation dataset attached.
#
# The packed train.bin is physically organized by source. Therefore,
# sequential sampling is NOT acceptable.
# ==============================================================================

from pathlib import Path

import torch
from torch.utils.data import RandomSampler

from transformers import Trainer


# ------------------------------------------------------------------------------
# Seed used exclusively for training-data ordering
# ------------------------------------------------------------------------------

DATA_SEED = SEED


# ------------------------------------------------------------------------------
# Explicit Trainer subclass
# ------------------------------------------------------------------------------
#
# We do not rely on the default sampler selected by the installed
# Transformers version.
#
# RandomSampler generates a global random permutation of the complete
# map-style dataset.
#
# A dedicated torch.Generator makes the permutation deterministic.
# ------------------------------------------------------------------------------

class MaiaTrainer(Trainer):

    def _get_train_sampler(self, train_dataset=None):

        if train_dataset is None:
            train_dataset = self.train_dataset

        if train_dataset is None:
            return None

        generator = torch.Generator()
        generator.manual_seed(DATA_SEED)

        return RandomSampler(
            train_dataset,
            replacement=False,
            generator=generator,
        )


# ------------------------------------------------------------------------------
# Build Trainer
# ------------------------------------------------------------------------------

trainer = MaiaTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)


# ------------------------------------------------------------------------------
# Locate latest FULL recovery checkpoint
# ------------------------------------------------------------------------------

checkpoints = []

for path in CHECKPOINT_DIR.glob("checkpoint-*"):

    try:
        step = int(
            path.name.split("-")[-1]
        )
    except ValueError:
        continue

    required_resume_files = [
        path / "trainer_state.json",
        path / "optimizer.pt",
        path / "scheduler.pt",
    ]

    if all(
        required_file.exists()
        for required_file
        in required_resume_files
    ):
        checkpoints.append(
            (
                step,
                path,
            )
        )


checkpoints.sort(
    key=lambda item: item[0]
)


if checkpoints:

    last_checkpoint = str(
        checkpoints[-1][1]
    )

    print(
        "Latest FULL recovery checkpoint:",
        last_checkpoint,
    )

else:

    last_checkpoint = None

    print(
        "No full recovery checkpoint found: "
        "training will start from scratch."
    )


# ------------------------------------------------------------------------------
# Audit the sampler returned DIRECTLY by MaiaTrainer
# ------------------------------------------------------------------------------

direct_sampler = trainer._get_train_sampler(
    train_dataset
)

print(
    "Direct sampler:",
    type(direct_sampler).__name__,
)

assert isinstance(
    direct_sampler,
    RandomSampler,
), (
    "MaiaTrainer did not return RandomSampler."
)

assert (
    direct_sampler.replacement
    is False
), (
    "Training sampler must sample without replacement."
)


# ------------------------------------------------------------------------------
# Determinism test
# ------------------------------------------------------------------------------
#
# Construct two independent samplers with the same seed.
# Their initial indices must be identical.
# ------------------------------------------------------------------------------

sampler_a = trainer._get_train_sampler(
    train_dataset
)

sampler_b = trainer._get_train_sampler(
    train_dataset
)

audit_count = 32

indices_a = []

indices_b = []

iterator_a = iter(sampler_a)
iterator_b = iter(sampler_b)

for _ in range(audit_count):

    indices_a.append(
        next(iterator_a)
    )

    indices_b.append(
        next(iterator_b)
    )


assert (
    indices_a
    ==
    indices_b
), (
    "Random sampler is not deterministic."
)


assert (
    len(set(indices_a))
    ==
    audit_count
), (
    "Unexpected duplicate indices in "
    "RandomSampler audit."
)


print(
    "Deterministic sampler: PASS"
)


# ------------------------------------------------------------------------------
# Global-range sanity check
# ------------------------------------------------------------------------------

dataset_length = len(
    train_dataset
)

assert (
    dataset_length
    ==
    TARGET_STEPS *
    EFFECTIVE_BATCH
), (
    "Training dataset length mismatch."
)


assert all(
    0 <= index < dataset_length
    for index
    in indices_a
), (
    "Sampler produced an invalid dataset index."
)


print(
    "Global index range:    PASS"
)


# ------------------------------------------------------------------------------
# Build actual training DataLoader
# ------------------------------------------------------------------------------

train_dataloader = (
    trainer.get_train_dataloader()
)


# ------------------------------------------------------------------------------
# Important:
#
# Accelerate / Trainer may wrap the original sampler when constructing
# the final DataLoader. Therefore the top-level DataLoader sampler is
# NOT used as the authoritative audit.
#
# We inspect it only for diagnostics.
# ------------------------------------------------------------------------------

visible_sampler = getattr(
    train_dataloader,
    "sampler",
    None,
)

print(
    "DataLoader sampler:",
    type(visible_sampler).__name__,
)


batch_sampler = getattr(
    train_dataloader,
    "batch_sampler",
    None,
)

print(
    "Batch sampler:",
    type(batch_sampler).__name__
    if batch_sampler is not None
    else "None",
)


nested_sampler = getattr(
    batch_sampler,
    "sampler",
    None,
)

print(
    "Nested sampler:",
    type(nested_sampler).__name__
    if nested_sampler is not None
    else "None",
)


# ------------------------------------------------------------------------------
# Dataset identity
# ------------------------------------------------------------------------------

assert (
    trainer.train_dataset
    is
    train_dataset
)

assert (
    trainer.eval_dataset
    is
    eval_dataset
)


print(
    "Training dataset:      PASS"
)

print(
    "Frozen eval dataset:   PASS"
)


# ------------------------------------------------------------------------------
# Resume semantics
# ------------------------------------------------------------------------------

assert (
    training_args.ignore_data_skip
    is False
), (
    "ignore_data_skip MUST remain False "
    "for deterministic resume."
)


if last_checkpoint is not None:

    checkpoint_path = Path(
        last_checkpoint
    )

    required_resume_files = [
        checkpoint_path /
        "trainer_state.json",

        checkpoint_path /
        "optimizer.pt",

        checkpoint_path /
        "scheduler.pt",
    ]

    for required_file in required_resume_files:

        assert (
            required_file.exists()
        ), (
            f"Missing recovery file: "
            f"{required_file}"
        )

    print(
        "Full checkpoint resume: PASS"
    )

else:

    print(
        "Full checkpoint resume: "
        "N/A - clean start"
    )


# ------------------------------------------------------------------------------
# Final report
# ------------------------------------------------------------------------------

print()
print("=" * 80)
print(
    "MAIA LITE - TRAINER AUDIT"
)
print("=" * 80)

print(
    f"Dataset sequences:      "
    f"{dataset_length:,}"
)

print(
    f"Global shuffle seed:    "
    f"{DATA_SEED}"
)

print(
    "Sampling:               "
    "GLOBAL RANDOM / WITHOUT REPLACEMENT"
)

print(
    "Sampler determinism:    PASS"
)

print(
    "Resume data skipping:   ENABLED"
)

print(
    "Validation attached:    PASS"
)

print(
    "Trainer construction:   PASS"
)

print("=" * 80)

print()
print(
    "TRAINING HAS NOT STARTED."
)
print(
    "Review this output before executing "
    "the start cell."
)

Latest FULL recovery checkpoint: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/checkpoints/checkpoint-216567
Direct sampler: RandomSampler
Deterministic sampler: PASS
Global index range:    PASS
DataLoader sampler: SequentialSampler
Batch sampler: BatchSampler
Nested sampler: SeedableRandomSampler
Training dataset:      PASS
Frozen eval dataset:   PASS
Full checkpoint resume: PASS

MAIA LITE - TRAINER AUDIT
Dataset sequences:      6,930,144
Global shuffle seed:    42
Sampling:               GLOBAL RANDOM / WITHOUT REPLACEMENT
Sampler determinism:    PASS
Resume data skipping:   ENABLED
Validation attached:    PASS
Trainer construction:   PASS

TRAINING HAS NOT STARTED.
Review this output before executing the start cell.


In [ ]:
# ==============================================================================
# 12. START / RESUME DEFINITIVE PRETRAINING
# ==============================================================================
#
# Execute ONLY after cells 10 and 11 pass on the A100.
#
# Full-state resume restores model, optimizer, scheduler, Trainer state and RNG.
# ignore_data_skip remains False.
# ==============================================================================

print("Starting definitive Maia Lite pretraining.")
print("Resume checkpoint:", last_checkpoint)

train_result = trainer.train(
    resume_from_checkpoint=last_checkpoint
)

print(train_result)

Starting definitive Maia Lite pretraining.
Resume checkpoint: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/checkpoints/checkpoint-216567


[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Step,Training Loss


TrainOutput(global_step=216567, training_loss=0.0, metrics={'train_runtime': 0.0051, 'train_samples_per_second': 1362606914.484, 'train_steps_per_second': 42581466.078, 'total_flos': 1.2872059091958104e+19, 'train_loss': 0.0, 'epoch': 1.0})


## Execution policy

**Normal CPU restart:** run cells 0–9 and Final Preflight. Existing tokenizer samples, tokenizer, corpus, and validation are read-only/frozen.

**A100 training runtime:** after Final Preflight, run cells 10–12. Cell 11 builds/audits the Trainer but does not train; cell 12 is the explicit start/resume gate.

Recommended behavioral checkpoints: 1k, 5k, 10k, 25k, 50k, 100k, 150k, 200k, and 216,567.
